In [1]:
import pandas as pd
import plotly.express as px
import planning

In [2]:
# Tennis parameters

CLASSEMENT_JOUEUR = "15/5"

PONDERATION_JOURS = {
    "lundi": 1,
    "mardi": 1,
    "mercredi": 1,
    "jeudi": 1,
    "vendredi": 1,
    "samedi": 1.5,
    "dimanche": 1.5,
    "ferie": 1.5,
}

In [3]:
# Display parameters

TOURNAMENT_COLOR = "grey"
PARTICIPATION_COLOR = "royalblue"

DATE_FORMAT = "%d/%m"

TEXT_COLOR = "white"

HOVER_DATA = {
    "debut": False,
    "fin": False,
    "club": False,
    "entree_reelle": True,
    "classement_min": True,
    "classement_max": True,
    "tours_avant_entree": True,
}

In [4]:
tournois = [
    {
        "club": "St Maurice",
        "debut": "2025-06-13",
        "fin": "2025-06-30",
        "entree_reelle" : "2025-06-18",
        "classement_min": "NC",
        "classement_max": "3/6"
    },
    {
        "club": "Plessis Trevise",
        "debut": "2025-06-30",
        "fin": "2025-07-16",
        "entree_reelle" : "2025-07-06",
        "classement_min": "NC",
        "classement_max": "15/1"
    },
    {
        "club": "Villecresnes",
        "debut": "2025-07-11",
        "fin": "2025-07-27",
        "entree_reelle" : "2025-07-18",
        "classement_min": "NC",
        "classement_max": "-15"
    }
]

In [5]:
df = pd.DataFrame(tournois)

df

,club,debut,fin,entree_reelle,classement_min,classement_max
0,St Maurice,2025-06-13,2025-06-30,2025-06-18,NC,3/6
1,Plessis Trevise,2025-06-30,2025-07-16,2025-07-06,NC,15/1
2,Villecresnes,2025-07-11,2025-07-27,2025-07-18,NC,-15


In [6]:
df["tours_avant_entree"] = df.apply(
    lambda row: planning.nombre_tours_avant_entree(
        row["classement_min"],
        row["classement_max"],
        CLASSEMENT_JOUEUR
    ),
    axis=1,
)
df["debut_participation"] = df.apply(
    lambda row: planning.date_entree_ponderee(
        row["debut"],
        row["tours_avant_entree"],
        PONDERATION_JOURS,
    ),
    axis=1,
)

fig = px.timeline(
    df,
    x_start="debut",
    x_end="fin",
    y="club",
    hover_data=HOVER_DATA,
)
fig.update_traces(marker_color=TOURNAMENT_COLOR)

fig_participation = px.timeline(
    df,
    x_start="debut_participation",
    x_end="fin",
    y="club",
)
fig_participation.update_traces(marker_color=PARTICIPATION_COLOR)

fig.add_traces(fig_participation.data)

df["entree_reelle"] = pd.to_datetime(df["entree_reelle"])

for _, row in df.iterrows():
    fig.add_shape(
        type="line",
        x0=row["entree_reelle"],
        x1=row["entree_reelle"],
        y0=row["club"],
        y1=row["club"],
        y0shift=-0.35,
        y1shift=0.35,
        xref="x",
        yref="y",
        line={"color": "red", "width": 3},
    )

    fig.add_annotation(
        x=row["debut"],
        y=row["club"],
        text=pd.to_datetime(row["debut"]).strftime(DATE_FORMAT),
        showarrow=False,
        xanchor="left",
        font={"color": TEXT_COLOR},
    )

    fig.add_annotation(
        x=row["fin"],
        y=row["club"],
        text=pd.to_datetime(row["fin"]).strftime(DATE_FORMAT),
        showarrow=False,
        xanchor="right",
        font={"color": TEXT_COLOR},
    )

    fig.add_annotation(
        x=row["debut_participation"],
        y=row["club"],
        text=pd.to_datetime(row["debut_participation"]).strftime(DATE_FORMAT),
        showarrow=False,
        xanchor="left",
        font={"color": TEXT_COLOR},
    )

fig.update_yaxes(autorange="reversed")

DATE_DEBUT_AFFICHAGE = "2025-06-08"
DATE_FIN_AFFICHAGE = "2025-07-31"

fig.update_xaxes(
    range=[DATE_DEBUT_AFFICHAGE, DATE_FIN_AFFICHAGE]
)
fig.show()